# 3팀 모델 비교 (10개 모델)


## 셀 1 — 설정 + imports + 한글 폰트

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

RESULTS_DIR = Path(r'D:\pill_detection\eval\results\_model_collection')

# 한글 폰트
for name in ['Malgun Gothic', '맑은 고딕', 'NanumGothic']:
    if name in {f.name for f in fm.fontManager.ttflist}:
        plt.rcParams['font.family'] = name
        break
plt.rcParams['axes.unicode_minus'] = False

# 분석 대상: exp_name → (표시ID, 짧은 라벨, 프레임워크, 담당)
MODELS = {
    'exp_faster_rcnn':  ('M1',  'Faster R-CNN', 'torchvision',     '승원'),
    'exp_rfdetr':       ('M2',  'RF-DETR',      'rfdetr',          '승원'),
    'exp_retinanet':    ('M3',  'RetinaNet',    'torchvision',     '수민'),
    'exp_ssd':          ('M4',  'SSD',          'torchvision',     '수민'),
    'exp_yolo11':       ('M5',  'YOLO11',       'ultralytics',     '지우'),
    'exp_yolo26':       ('M6',  'YOLO26',       'ultralytics',     '지우'),
    'exp_yolo9':        ('M7',  'YOLO9',        'ultralytics',     '수빈'),
    'exp_rtdetr':       ('M8',  'RT-DETR',      'ultralytics',     '수빈'),
    'exp_sam':          ('M9',  'SAM',          'SAM',             '귀한'),
    'exp_yolo11_seg':   ('M10', 'YOLO11-seg',   'ultralytics-seg', '귀한'),
}

# 1차 필터 기준 (설계서 v3)
DETECTION_RATE_MIN = 0.8       # 33장 중 27장 이상 탐지
MAX_MISSED = 5                 # 미탐지 5장 이하

# 분석 제외 (구버전 / 잘못 들어온 파일)
EXCLUDE_EXP_NAMES = set()

# 프레임워크별 색상 (시각화 일관성)
FW_COLORS = {
    'torchvision':     '#3498db',
    'rfdetr':          '#e74c3c',
    'ultralytics':     '#2ecc71',
    'ultralytics-seg': '#9b59b6',
    'SAM':             '#e67e22',
}

assert RESULTS_DIR.exists(), f'폴더 없음: {RESULTS_DIR}'
print(f'결과 폴더: {RESULTS_DIR}')
print(f'분석 대상 모델: {len(MODELS)}개')
print(f'1차 필터: Detection rate ≥ {DETECTION_RATE_MIN} & 미탐지 ≤ {MAX_MISSED}')


## 셀 2 — JSON 로드 + 이름 기반 재채점 + 분류 + 1차 필터


In [ ]:
def rescore(data):
    # 이름 기반 재채점 (class_idx 매핑 버그 회피)
    n = len(data['results'])
    top1_hit = 0
    top3_hit = 0
    detected = 0
    for r in data['results']:
        gt_name = r.get('gt_dl_name')
        if not gt_name or r.get('num_detections', 0) == 0:
            continue
        detected += 1
        top1_names = [d['top3_class_name'][0] for d in r['detections'] if d.get('top3_class_name')]
        top3_names = [nm for d in r['detections'] for nm in d.get('top3_class_name', [])]
        if gt_name in top1_names:
            top1_hit += 1
        if gt_name in top3_names:
            top3_hit += 1
    data['detection_rate'] = round(detected / n, 4) if n else 0.0
    data['top1_accuracy']  = round(top1_hit / n, 4)  if n else 0.0
    data['top3_accuracy']  = round(top3_hit / n, 4)  if n else 0.0
    data['_n_detected']    = detected
    data['_n_missed']      = n - detected
    return data


def classify(data):
    # exp_name 기반 분류. 제외 대상 / 10개 모델 / 미분류 구분.
    name = data.get('exp_name', '')

    # 제외 대상
    if name in EXCLUDE_EXP_NAMES:
        return {'_include': False, '_exp_id': name, '_short': name,
                '_framework': 'Excluded', '_author': '-'}

    # 10개 모델
    if name in MODELS:
        exp_id, short, fw, author = MODELS[name]
        return {'_include': True, '_exp_id': exp_id, '_short': short,
                '_framework': fw, '_author': author}

    # 매핑 없음 (새 모델 추가 시 MODELS 에 등록 필요)
    return {'_include': False, '_exp_id': '?', '_short': name,
            '_framework': 'Unknown', '_author': '-'}


def passes_filter(data):
    # 1차 필터: Detection rate ≥ DETECTION_RATE_MIN & 미탐지 ≤ MAX_MISSED
    return (data['detection_rate'] >= DETECTION_RATE_MIN
            and data['_n_missed'] <= MAX_MISSED)


jsons = sorted(RESULTS_DIR.glob('*.json'))
# 자동 생성 파일 제외
jsons = [j for j in jsons if not j.name.startswith(('comparison_', 'fig'))]
assert len(jsons) > 0, f'JSON 없음: {RESULTS_DIR}'

all_data = []
for j in jsons:
    with open(j, 'r', encoding='utf-8') as f:
        data = json.load(f)
    data = rescore(data)
    data.update(classify(data))
    data['_passes_filter'] = passes_filter(data) if data['_include'] else False
    data['_source_file'] = j.name
    all_data.append(data)

# 출력
print(f"{'포함':4s} | {'1차':4s} | {'ID':4s} | {'프레임워크':16s} | {'담당':4s} | "
      f"{'exp_name':22s} | Det / Top-1 / Top-3 (미탐지)")
print('-' * 110)
for d in all_data:
    inc_flag = '✅' if d['_include'] else '❌'
    filter_flag = '🟢' if d['_passes_filter'] else ('🔴' if d['_include'] else '-')
    print(f"{inc_flag}  | {filter_flag}  | {d['_exp_id']:4s} | {d['_framework']:16s} | "
          f"{d['_author']:4s} | {d.get('exp_name', '?'):22s} | "
          f"{d['detection_rate']:.3f} / {d['top1_accuracy']:.3f} / {d['top3_accuracy']:.3f} "
          f"({d['_n_missed']}장)")

# 검증
included = [d for d in all_data if d['_include']]
passed   = [d for d in included if d['_passes_filter']]

print()
print(f'포함: {len(included)}개 / 1차 필터 통과: {len(passed)}개')
print(f'제외: {len(all_data) - len(included)}개')

if len(passed) == 0:
    print('⚠️  1차 필터 통과 모델 없음 — 선발 불가')
elif len(passed) < 3:
    print(f'⚠️  1차 필터 통과 {len(passed)}개뿐 — Top-3 선발 어려움')


## 셀 3 — 비교표 생성 + 저장

포함 대상만 테이블로. 정렬: Baseline → B1~B3 → C1~C3 → D1~D2 → E1~E2 → F


In [ ]:
# 정렬 순서
ID_ORDER = ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'M10']

rows = []
for d in all_data:
    if not d['_include']:
        continue
    rows.append({
        'id':             d['_exp_id'],
        'short':          d['_short'],
        'framework':      d['_framework'],
        'author':         d['_author'],
        'exp_name':       d.get('exp_name', ''),
        'num_images':     d.get('num_images', 0),
        'detection_rate': d['detection_rate'],
        'n_missed':       d['_n_missed'],
        'top1_accuracy':  d['top1_accuracy'],
        'top3_accuracy':  d['top3_accuracy'],
        'passes_filter':  d['_passes_filter'],
    })

df = pd.DataFrame(rows)
df['_order'] = df['id'].apply(lambda x: ID_ORDER.index(x) if x in ID_ORDER else 999)
df = df.sort_values('_order').drop('_order', axis=1).reset_index(drop=True)

# 저장
df.to_csv(RESULTS_DIR / 'comparison_table.csv', index=False, encoding='utf-8-sig')
try:
    md_content = '# 모델 비교표 (10개 모델)\n\n' + df.to_markdown(index=False)
except ImportError:
    md_content = '# 모델 비교표 (10개 모델)\n\n```\n' + df.to_string(index=False) + '\n```'

with open(RESULTS_DIR / 'comparison_table.md', 'w', encoding='utf-8') as f:
    f.write(md_content)

print('저장: comparison_table.csv / .md\n')
df


## 셀 4 — 시각화 1: 모델별 성능 막대


In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.5))
x = np.arange(len(df))
w = 0.27

bars_det = ax.bar(x - w, df['detection_rate'], w, label='Detection', color='#4a90d9', alpha=0.85)
bars_t1  = ax.bar(x,     df['top1_accuracy'],  w, label='Top-1',     color='#50c878', alpha=0.85)
bars_t3  = ax.bar(x + w, df['top3_accuracy'],  w, label='Top-3',     color='#f5a623', alpha=0.85)

for i, (det, t1, t3, passed) in enumerate(zip(df['detection_rate'], df['top1_accuracy'],
                                               df['top3_accuracy'], df['passes_filter'])):
    ax.text(i - w, det + 0.01, f'{det:.2f}', ha='center', fontsize=7.5)
    ax.text(i,     t1  + 0.01, f'{t1:.2f}',  ha='center', fontsize=7.5)
    ax.text(i + w, t3  + 0.01, f'{t3:.2f}',  ha='center', fontsize=7.5)
    # 1차 필터 탈락 표시
    if not passed:
        ax.text(i, 1.10, '🔴', ha='center', fontsize=11)

# x축 라벨 = ID + short (예: "M1\nFaster R-CNN")
labels = [f"{r.id}\n{r.short}" for r in df.itertuples()]
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylim(0, 1.18)
ax.set_ylabel('Accuracy')
ax.set_title(f'모델별 성능 비교 ({df["num_images"].iloc[0]}장 평가셋, 🔴 = 1차 필터 탈락)')
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)

# 프레임워크별 x축 배경색 블록
prev_fw = None
start = 0
for i, fw in enumerate(df['framework']):
    if fw != prev_fw and prev_fw is not None:
        color = FW_COLORS.get(prev_fw, '#cccccc')
        ax.axvspan(start - 0.5, i - 0.5, alpha=0.05, color=color, zorder=0)
        start = i
    prev_fw = fw
# 마지막 블록
if prev_fw is not None:
    color = FW_COLORS.get(prev_fw, '#cccccc')
    ax.axvspan(start - 0.5, len(df) - 0.5, alpha=0.05, color=color, zorder=0)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig1_accuracy_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: fig1_accuracy_bars.png')


## 셀 5 — 시각화 2: 이미지별 적중 히트맵 (모델별)


In [ ]:
included = [d for d in all_data if d['_include']]
# df 순서와 맞추기
included_by_id = {d['_exp_id']: d for d in included}
included_sorted = [included_by_id[i] for i in df['id']]

filenames_set = set()
for d in included_sorted:
    for r in d['results']:
        filenames_set.add(r['filename'])
filenames = sorted(filenames_set)

matrix = np.full((len(filenames), len(included_sorted)), np.nan)
for col, d in enumerate(included_sorted):
    for r in d['results']:
        row = filenames.index(r['filename'])
        gt_name = r.get('gt_dl_name')
        if r.get('num_detections', 0) == 0:
            matrix[row, col] = 0
            continue
        top1_names = [dd['top3_class_name'][0] for dd in r['detections'] if dd.get('top3_class_name')]
        top3_names = [nm for dd in r['detections'] for nm in dd.get('top3_class_name', [])]
        if gt_name in top1_names:
            matrix[row, col] = 2
        elif gt_name in top3_names:
            matrix[row, col] = 1
        else:
            matrix[row, col] = 0

fig, ax = plt.subplots(figsize=(max(8, len(included_sorted) * 0.85), max(8, len(filenames) * 0.28)))
cmap = plt.matplotlib.colors.ListedColormap(['#d9534f', '#f0ad4e', '#5cb85c'])
ax.imshow(matrix, cmap=cmap, aspect='auto', vmin=0, vmax=2)

x_labels = [f"{r.id}\n{r.short}" for r in df.itertuples()]
ax.set_xticks(range(len(included_sorted)))
ax.set_xticklabels(x_labels, fontsize=8)
ax.set_yticks(range(len(filenames)))
ax.set_yticklabels([Path(f).stem[:30] for f in filenames], fontsize=7)

from matplotlib.patches import Patch
handles = [
    Patch(color='#5cb85c', label='Top-1 적중'),
    Patch(color='#f0ad4e', label='Top-3 적중'),
    Patch(color='#d9534f', label='오답/미탐지'),
]
ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.04), ncol=3, fontsize=9)
ax.set_title('이미지별 적중 히트맵 (모델별)')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig2_hit_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: fig2_hit_heatmap.png')


## 셀 6 — 시각화 3: 모델별 Top-1 Confidence 분포


In [ ]:
data_list = []
labels = []
colors_list = []
for d in included_sorted:
    confs = []
    for r in d['results']:
        for det in r.get('detections', []):
            if det.get('top3_confidence'):
                confs.append(det['top3_confidence'][0])
    if confs:
        data_list.append(confs)
        labels.append(f"{d['_exp_id']}\n{d['_short']}")
        colors_list.append(FW_COLORS.get(d['_framework'], '#888888'))

fig, ax = plt.subplots(figsize=(13, 5.5))
bp = ax.boxplot(data_list, labels=labels, patch_artist=True, widths=0.55)

for patch, c in zip(bp['boxes'], colors_list):
    patch.set_facecolor(c)
    patch.set_alpha(0.55)

plt.xticks(fontsize=9)
ax.set_ylabel('Top-1 Confidence')
ax.set_title('모델별 Top-1 Confidence 분포 (box 색 = 프레임워크)')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig3_confidence_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: fig3_confidence_boxplot.png')


## 셀 7 — 시각화 4: 자주 발생하는 오분류 Top-15


In [ ]:
confusion = {}
for d in included_sorted:
    for r in d['results']:
        gt = r.get('gt_dl_name')
        if not gt or r.get('num_detections', 0) == 0:
            continue
        for det in r['detections']:
            if not det.get('top3_class_name'):
                continue
            pred = det['top3_class_name'][0]
            if pred != gt:
                key = (gt, pred)
                confusion[key] = confusion.get(key, 0) + 1

if not confusion:
    print('오답 없음 — 스킵')
else:
    top_conf = sorted(confusion.items(), key=lambda x: -x[1])[:15]
    labels_c = [f'{gt[:15]} → {pred[:15]}' for (gt, pred), _ in top_conf]
    counts   = [c for _, c in top_conf]

    fig, ax = plt.subplots(figsize=(10, 6))
    y = np.arange(len(labels_c))
    ax.barh(y, counts, color='#d9534f', alpha=0.75)
    ax.set_yticks(y)
    ax.set_yticklabels(labels_c, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('오답 횟수 (전체 모델 합산)')
    ax.set_title('자주 발생하는 오분류 Top-15 (10개 모델 합산)')
    ax.grid(axis='x', alpha=0.3)
    for i, c in enumerate(counts):
        ax.text(c + 0.05, i, str(c), va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'fig4_failure_confusion.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('저장: fig4_failure_confusion.png')


## 셀 8 — 시각화 5: 모델별 절대 성능 비교 (Top-1 내림차순)


In [ ]:
# 1차 필터 통과 / 탈락 분리
passed_df  = df[df['passes_filter']].copy()
failed_df  = df[~df['passes_filter']].copy()

# Top-1 내림차순 정렬 (전체 — 통과/탈락 모두 포함하되 메달은 통과만)
sorted_df = df.sort_values('top1_accuracy', ascending=False).reset_index(drop=True)

# 메달 (1차 필터 통과 모델 중 Top-1 상위 3개)
medals = {}
passed_sorted = passed_df.sort_values('top1_accuracy', ascending=False).reset_index(drop=True)
if len(passed_sorted) >= 1: medals[passed_sorted.iloc[0]['id']] = '🥇'
if len(passed_sorted) >= 2: medals[passed_sorted.iloc[1]['id']] = '🥈'
if len(passed_sorted) >= 3: medals[passed_sorted.iloc[2]['id']] = '🥉'

fig, ax = plt.subplots(figsize=(13, 5.5))
x = np.arange(len(sorted_df))
w = 0.38

# 색상: 1차 필터 탈락이면 회색
def get_color(passed, base_color, fail_color='#cccccc'):
    return [base_color if p else fail_color for p in passed]

t1_colors = get_color(sorted_df['passes_filter'], '#50c878')
t3_colors = get_color(sorted_df['passes_filter'], '#f5a623')

bars1 = ax.bar(x - w/2, sorted_df['top1_accuracy'], w, label='Top-1', color=t1_colors)
bars2 = ax.bar(x + w/2, sorted_df['top3_accuracy'], w, label='Top-3', color=t3_colors)

# 값 + 메달 표기
for i, (t1, t3, eid) in enumerate(zip(sorted_df['top1_accuracy'],
                                       sorted_df['top3_accuracy'],
                                       sorted_df['id'])):
    medal = medals.get(eid, '')
    ax.text(i - w/2, t1 + 0.008, f'{medal}{t1:.3f}', ha='center', fontsize=8)
    ax.text(i + w/2, t3 + 0.008, f'{t3:.3f}',         ha='center', fontsize=8)

ax.set_xticks(x)
x_labels_sorted = [f"{r.id}\n{r.short}" for r in sorted_df.itertuples()]
ax.set_xticklabels(x_labels_sorted, fontsize=9)
ax.set_ylabel('Accuracy')
ax.set_title('모델별 절대 성능 비교 (Top-1 내림차순 / 회색 = 1차 필터 탈락 / 🥇🥈🥉 = 통과 중 상위 3개)')
ax.set_ylim(0, max(sorted_df['top3_accuracy'].max() * 1.15, 0.5))
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig5_model_ranking.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: fig5_model_ranking.png')


## 셀 9 — 최종 모델 Top-3 선발 (1차 필터 → Top-1 → Top-3 tie-break)


In [ ]:
print('=' * 70)
print(f'모델 비교 완료 — {RESULTS_DIR}')
print('=' * 70)

# === 1차 필터 통과 현황 ===
print()
print('1차 필터 (Detection rate ≥ 0.8 & 미탐지 ≤ 5장)')
print('-' * 70)
passed_df = df[df['passes_filter']].copy()
failed_df = df[~df['passes_filter']].copy()

print(f'통과: {len(passed_df)}/{len(df)}개')
if len(failed_df) > 0:
    print('탈락 모델:')
    for _, row in failed_df.iterrows():
        print(f'  ❌ {row["id"]:4s} {row["short"]:14s} — '
              f'Det {row["detection_rate"]:.3f} / 미탐지 {row["n_missed"]}장')

# === 모델 선발 ===
print()
if len(passed_df) == 0:
    print('⚠️  1차 필터 통과 모델 없음 — 선발 불가. 학습 조건 재검토 필요.')
elif len(passed_df) < 3:
    print(f'⚠️  1차 필터 통과 {len(passed_df)}개뿐 — Top-3 미만이지만 가능한 만큼 선발')

if len(passed_df) > 0:
    # 2차: Top-1 내림차순
    rank = passed_df.sort_values(['top1_accuracy', 'top3_accuracy'],
                                 ascending=[False, False]).reset_index(drop=True)

    # 3차 tie-break: Top-1 ±6%p (±2장) 이내면 Top-3로 재정렬
    TIE_BREAK_THRESHOLD = 0.06   # ±6%p (33장 기준 ±2장)

    top1_best = rank.iloc[0]['top1_accuracy']
    in_tie_zone = rank[abs(rank['top1_accuracy'] - top1_best) <= TIE_BREAK_THRESHOLD].copy()
    out_of_tie  = rank[abs(rank['top1_accuracy'] - top1_best) >  TIE_BREAK_THRESHOLD].copy()

    if len(in_tie_zone) > 1:
        print(f'🔍 Top-1 동률 구간 ({len(in_tie_zone)}개, ±{TIE_BREAK_THRESHOLD*100:.0f}%p) — Top-3로 tie-break')
        in_tie_zone = in_tie_zone.sort_values(['top3_accuracy', 'top1_accuracy'],
                                              ascending=[False, False])

    final_rank = pd.concat([in_tie_zone, out_of_tie]).reset_index(drop=True)

    # === Top-3 선발 ===
    print()
    print('🎯 모델 Top-3 선발 (1차 필터 통과 → Top-1 → Top-3 tie-break)')
    print('-' * 70)
    top_n = min(3, len(final_rank))
    for rank_idx in range(top_n):
        row = final_rank.iloc[rank_idx]
        medal = ['🥇', '🥈', '🥉'][rank_idx]
        print(f'{medal} {row["id"]:4s} {row["short"]:16s} ({row["framework"]:16s}, {row["author"]}) '
              f'Top-1 {row["top1_accuracy"]:.3f} / Top-3 {row["top3_accuracy"]:.3f} '
              f'/ Det {row["detection_rate"]:.3f}')

    # === 4차 안내 (수동) ===
    print()
    print('📋 4차 단계 (재현성 검증) — 수동 진행 필요')
    print('-' * 70)
    print('  Top-3 모델을 seed=100으로 재학습 1회 → seed=42와 모두 상위권이어야 통과')
    print('  → 5차 최종: 4차 통과 모델 중 2회 Top-1 평균 최고')

# === 1차 필터 통과 모델 전체 랭킹 ===
if len(passed_df) > 0:
    print()
    print('1차 필터 통과 모델 전체 랭킹 (Top-1 내림차순)')
    print('-' * 70)
    for i, row in final_rank.iterrows():
        in_tie = '⚠️' if abs(row['top1_accuracy'] - top1_best) <= TIE_BREAK_THRESHOLD and i < len(in_tie_zone) else '  '
        print(f'  {i+1:2d}. {in_tie} {row["id"]:4s} {row["short"]:16s} '
              f'Top-1 {row["top1_accuracy"]:.3f}  Top-3 {row["top3_accuracy"]:.3f}  '
              f'({row["framework"]:16s}, {row["author"]})')

# 생성 파일 목록
print()
print('생성 파일:')
for f in sorted(RESULTS_DIR.glob('*')):
    if f.name.startswith(('comparison_', 'fig')):
        print(f'  - {f.name}')
